<a href="https://colab.research.google.com/github/m-zayed5722/Miscellaneous-Projects/blob/main/GenAI_Platform_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
{
    "allowed": True,
    "confidence": 0.94,
    "warnings": [],
    "redacted_text": "...",
}

{'allowed': True, 'confidence': 0.94, 'warnings': [], 'redacted_text': '...'}

In [2]:
TOOLS = {
    ...
}

In [ ]:
# ============================================================
# PromptBench Lite
# Automatic Prompt Benchmarking
# Runs entirely in Google Colab
# ============================================================

!pip -q install transformers accelerate bitsandbytes pandas tqdm

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from tqdm import tqdm



In [ ]:
# ------------------------------------------------------------
# Load Model
# ------------------------------------------------------------

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("Model loaded.\n")


In [ ]:
# ------------------------------------------------------------
# Benchmark Dataset
# ------------------------------------------------------------

dataset = [

{
    "question":"What is the capital of Canada?",
    "answer":"Ottawa"
},

{
    "question":"What planet is known as the Red Planet?",
    "answer":"Mars"
},

{
    "question":"Who wrote Hamlet?",
    "answer":"William Shakespeare"
},

{
    "question":"What is 12 * 8?",
    "answer":"96"
},

{
    "question":"Largest ocean on Earth?",
    "answer":"Pacific Ocean"
}

]


In [ ]:

# ------------------------------------------------------------
# Prompt Templates
# ------------------------------------------------------------

templates = {

"Simple":

"""Answer the following question.

Question:
{question}

Answer:
""",

"Detailed":

"""You are an expert assistant.

Provide the most accurate answer possible.

Question:
{question}

Answer:
""",

"Step-by-Step":

"""Solve carefully.

Question:
{question}

Think briefly before answering.

Final Answer:
""",

"Short":

"""Q:
{question}

A:
"""

}

In [4]:


# ------------------------------------------------------------
# Generation Function
# ------------------------------------------------------------

def generate(prompt):

    output = generator(
        prompt,
        max_new_tokens=50,
        do_sample=False,
        temperature=0.0,
    )[0]["generated_text"]

    return output[len(prompt):].strip()

# ------------------------------------------------------------
# Simple Evaluator
# ------------------------------------------------------------

def score(prediction, truth):

    prediction = prediction.lower()

    truth = truth.lower()

    if truth in prediction:
        return 1

    return 0

# ------------------------------------------------------------
# Run Benchmark
# ------------------------------------------------------------

results = []

for template_name, template in templates.items():

    total = 0

    print(f"\nRunning template: {template_name}")

    for item in tqdm(dataset):

        prompt = template.format(question=item["question"])

        prediction = generate(prompt)

        s = score(prediction, item["answer"])

        total += s

        results.append({

            "Template":template_name,
            "Question":item["question"],
            "Prediction":prediction,
            "Expected":item["answer"],
            "Correct":s

        })

# ------------------------------------------------------------
# Results DataFrame
# ------------------------------------------------------------

df = pd.DataFrame(results)

display(df)

# ------------------------------------------------------------
# Leaderboard
# ------------------------------------------------------------

leaderboard = (

df.groupby("Template")["Correct"]
.mean()
.sort_values(ascending=False)
.reset_index()

)

leaderboard["Accuracy"] = leaderboard["Correct"] * 100

leaderboard = leaderboard.drop(columns="Correct")

print("\n==============================")
print("Prompt Leaderboard")
print("==============================")

display(leaderboard)

# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

df.to_csv("benchmark_results.csv", index=False)

leaderboard.to_csv("leaderboard.csv", index=False)

print("\nSaved:")
print("- benchmark_results.csv")
print("- leaderboard.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00
Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

KeyboardInterrupt: 

# PromptBench Lite

Automatically benchmark prompt templates against an open-source LLM.

## Features

- Multiple prompt templates
- Automatic benchmarking
- Accuracy leaderboard
- CSV export
- Google Colab compatible
- Open-source models only

## Model

- Microsoft Phi-3 Mini 4K Instruct

## Output

- benchmark_results.csv
- leaderboard.csv